# PTCG R117 — Manual Colab Launcher

这是 R117 Win-Guided Trajectory Distillation 的手动启动版。

运行前：

1. 在 **代码执行程序 → 更改运行时类型** 中选择 **T4 / L4 / A100 GPU**。
2. 依次运行全部单元格。
3. Notebook 会让你手动上传 `kaggle.json` 和
   `PTCG_R117_WIN_GUIDED_TRAJECTORY_DISTILLATION_CODE_PACKAGE_20260729.zip`。
4. 六天输入直接从 Kaggle 官方 daily episode datasets 重新构建；不会下载或调用
   R109XL、R115、R116、R103、R098、旧 checkpoint、旧 Agent、Search、MCTS 或 PPO。
5. 任一 Gate 失败后流水线会停止，不会降低阈值或继续调参。

预计：官方回放准备约 40–120 分钟，正式流水线通常约 2.5–6 小时。


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from google.colab import files
from pathlib import Path
import os

print('请选择你的 kaggle.json')
uploaded = files.upload()
key_names = [name for name in uploaded if Path(name).name == 'kaggle.json']
assert len(key_names) == 1, f'必须且只能上传一个 kaggle.json，实际：{list(uploaded)}'

kaggle_dir = Path('/root/.kaggle')
kaggle_dir.mkdir(parents=True, exist_ok=True)
key_path = kaggle_dir / 'kaggle.json'
key_path.write_bytes(uploaded[key_names[0]])
os.chmod(key_path, 0o600)
print('Kaggle key ready:', key_path)


In [ ]:
from google.colab import files
from pathlib import Path
import hashlib
import shutil

PACKAGE_NAME = 'PTCG_R117_WIN_GUIDED_TRAJECTORY_DISTILLATION_CODE_PACKAGE_20260729.zip'
PACKAGE_SHA256 = '8f7f5f681a3cb7ac5a1622c9268a1fadc349b79b9003c925f1b13b20a3685e98'

print('请选择 R117 Win-Guided 代码包：', PACKAGE_NAME)
uploaded = files.upload()
package_names = [name for name in uploaded if Path(name).name == PACKAGE_NAME]
assert len(package_names) == 1, f'上传文件不正确：{list(uploaded)}'
raw = uploaded[package_names[0]]
actual_sha = hashlib.sha256(raw).hexdigest()
assert actual_sha == PACKAGE_SHA256, (actual_sha, PACKAGE_SHA256)

package_dir = Path('/content/drive/MyDrive/PTCG/packages')
package_dir.mkdir(parents=True, exist_ok=True)
PACKAGE_ZIP = package_dir / PACKAGE_NAME
PACKAGE_ZIP.write_bytes(raw)

REPLAY_ROOT = Path('/content/drive/MyDrive/PTCG/replays/R117_OFFICIAL_6D_20260729')
RESULT_ROOT = Path('/content/drive/MyDrive/PTCG/results')
REPLAY_ROOT.mkdir(parents=True, exist_ok=True)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print('Package:', PACKAGE_ZIP)
print('Replay root:', REPLAY_ROOT)
print('Result root:', RESULT_ROOT)


In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'kaggle==2.2.4', 'orjson>=3.10'],
    check=True,
)
print('Kaggle API and replay builder dependencies ready.')


In [ ]:
from __future__ import annotations

import csv
import hashlib
import io
import json
import re
import shutil
import zipfile
from collections import Counter
from datetime import date, datetime, timedelta, timezone
from pathlib import Path, PurePosixPath

import orjson
from kaggle.api.kaggle_api_extended import KaggleApi

COMPETITION = 'pokemon-tcg-ai-battle'
DATASET_PREFIX = 'kaggle/pokemon-tcg-ai-battle-episodes-'
BUILD_ROOT = Path('/content/r117_official_replay_build')
BUILD_ROOT.mkdir(parents=True, exist_ok=True)
TEAM_NAMES_RE = re.compile(rb'"TeamNames"\s*:\s*(\[[^\]]*\])')
INDEX_FIELDS = [
    'date', 'episode_id', 'archive_member', 'module_version',
    'team0', 'team1', 'leaderboard_team0', 'leaderboard_team1',
    'top30_rank0', 'top30_rank1', 'reward0', 'reward1',
    'steps', 'raw_bytes',
]


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()


def latest_consecutive_six(api: KaggleApi) -> list[str]:
    rows = api.dataset_list(
        search='pokemon tcg ai battle episodes',
        sort_by='updated',
        page=1,
    ) or []
    pattern = re.compile(
        r'^kaggle/pokemon-tcg-ai-battle-episodes-(\d{4}-\d{2}-\d{2})$'
    )
    available: set[date] = set()
    for row in rows:
        match = pattern.fullmatch(str(getattr(row, 'ref', '') or ''))
        if match:
            available.add(date.fromisoformat(match.group(1)))
    for end in sorted(available, reverse=True):
        candidate = [end - timedelta(days=offset) for offset in range(5, -1, -1)]
        if all(day in available for day in candidate):
            return [day.isoformat() for day in candidate]
    raise RuntimeError(
        f'没有发现连续 6 个官方日期；可见日期：'
        f'{[d.isoformat() for d in sorted(available)]}'
    )


def current_top30(api: KaggleApi) -> tuple[dict[str, int], str]:
    api.competition_leaderboard_download(
        COMPETITION,
        path=str(BUILD_ROOT),
        quiet=False,
    )
    archive_path = BUILD_ROOT / f'{COMPETITION}.zip'
    with zipfile.ZipFile(archive_path) as archive:
        csv_members = [
            name for name in archive.namelist()
            if name.lower().endswith('.csv')
        ]
        if len(csv_members) != 1:
            raise RuntimeError(f'Unexpected leaderboard members: {csv_members}')
        snapshot = PurePosixPath(csv_members[0]).name
        with archive.open(csv_members[0]) as handle:
            rows = list(
                csv.DictReader(
                    io.TextIOWrapper(handle, encoding='utf-8-sig', newline='')
                )
            )
    top30: dict[str, int] = {}
    for row in rows:
        try:
            rank = int(float(str(row.get('Rank') or '')))
        except ValueError:
            continue
        team = str(row.get('TeamName') or '').strip()
        if 1 <= rank <= 30 and team:
            top30[team] = rank
    if len(top30) != 30:
        raise RuntimeError(f'Expected 30 leaderboard teams, found {len(top30)}')
    return top30, snapshot


def find_member(names: list[str], basename: str) -> str:
    matches = [name for name in names if PurePosixPath(name).name == basename]
    if len(matches) != 1:
        raise RuntimeError(f'Expected one {basename}, found {matches}')
    return matches[0]


def build_day(
    api: KaggleApi,
    day: str,
    rank_by_team: dict[str, int],
    leaderboard_snapshot: str,
) -> dict[str, object]:
    ref = DATASET_PREFIX + day
    day_root = BUILD_ROOT / day
    day_root.mkdir(parents=True, exist_ok=True)
    print(f'[{day}] downloading official dataset {ref}', flush=True)
    api.dataset_download_files(
        ref,
        path=str(day_root),
        force=False,
        quiet=False,
        unzip=False,
    )
    source_zip = day_root / f'pokemon-tcg-ai-battle-episodes-{day}.zip'
    if not source_zip.is_file():
        raise FileNotFoundError(source_zip)

    local_output = day_root / f'r117_top30_replays_{day}.zip'
    drive_output = REPLAY_ROOT / local_output.name
    index_rows: list[dict[str, object]] = []
    parse_failures = 0

    with zipfile.ZipFile(source_zip) as source:
        names = source.namelist()
        json_members = sorted(
            name for name in names
            if name.lower().endswith('.json')
            and PurePosixPath(name).name[:-5].isdigit()
        )
        print(
            f'[{day}] official episode JSON files={len(json_members)}',
            flush=True,
        )
        with zipfile.ZipFile(
            local_output,
            mode='w',
            compression=zipfile.ZIP_DEFLATED,
            compresslevel=1,
            allowZip64=True,
        ) as target:
            for scanned, source_member in enumerate(json_members, start=1):
                episode_id = int(PurePosixPath(source_member).stem)
                try:
                    with source.open(source_member) as handle:
                        prefix = handle.read(131072)
                    match = TEAM_NAMES_RE.search(prefix)
                    if match is None:
                        raise ValueError('TeamNames not found in prefix')
                    teams = [str(value) for value in orjson.loads(match.group(1))]
                    if len(teams) != 2:
                        raise ValueError(f'invalid TeamNames={teams}')
                except Exception as exc:
                    parse_failures += 1
                    print(f'[{day}] WARN {episode_id}: {exc}', flush=True)
                    continue

                ranks = [rank_by_team.get(teams[0]), rank_by_team.get(teams[1])]
                if ranks[0] is None and ranks[1] is None:
                    continue

                raw = source.read(source_member)
                episode = orjson.loads(raw)
                episode_teams = [
                    str(value) for value in episode['info']['TeamNames']
                ]
                if episode_teams != teams:
                    raise RuntimeError(f'team mismatch in episode {episode_id}')
                rewards = list(episode.get('rewards') or [0, 0])
                if len(rewards) != 2:
                    raise RuntimeError(f'invalid rewards in episode {episode_id}')

                member = f'{episode_id}.json'
                target.writestr(member, raw)
                index_rows.append(
                    {
                        'date': day,
                        'episode_id': episode_id,
                        'archive_member': member,
                        'module_version': str(
                            episode.get('module_version') or ''
                        ),
                        'team0': teams[0],
                        'team1': teams[1],
                        'leaderboard_team0': teams[0],
                        'leaderboard_team1': teams[1],
                        'top30_rank0': '' if ranks[0] is None else ranks[0],
                        'top30_rank1': '' if ranks[1] is None else ranks[1],
                        'reward0': rewards[0],
                        'reward1': rewards[1],
                        'steps': len(episode.get('steps') or []),
                        'raw_bytes': len(raw),
                    }
                )
                if len(index_rows) % 100 == 0:
                    print(
                        f'[{day}] selected={len(index_rows)} '
                        f'scanned={scanned}/{len(json_members)}',
                        flush=True,
                    )

            buffer = io.StringIO(newline='')
            writer = csv.DictWriter(buffer, fieldnames=INDEX_FIELDS)
            writer.writeheader()
            writer.writerows(index_rows)
            target.writestr(
                '_index.csv',
                buffer.getvalue().encode('utf-8'),
            )

    if not index_rows:
        raise RuntimeError(f'No Top30 replay rows selected for {day}')
    with zipfile.ZipFile(local_output) as audit:
        if '_index.csv' not in audit.namelist():
            raise RuntimeError(f'Missing _index.csv: {local_output}')
        json_count = sum(
            name.endswith('.json') for name in audit.namelist()
        )
        if json_count != len(index_rows):
            raise RuntimeError(
                f'Archive/index mismatch {json_count} != {len(index_rows)}'
            )

    shutil.copy2(local_output, drive_output)
    result = {
        'date': day,
        'source_dataset': ref,
        'leaderboard_snapshot': leaderboard_snapshot,
        'official_episode_jsons': len(json_members),
        'selected_episodes': len(index_rows),
        'parse_failures': parse_failures,
        'output': str(drive_output),
        'output_bytes': drive_output.stat().st_size,
        'sha256': sha256(drive_output),
        'module_versions': sorted(
            {str(row['module_version']) for row in index_rows}
        ),
    }
    print(f'[{day}] READY {json.dumps(result, ensure_ascii=False)}', flush=True)

    source_zip.unlink(missing_ok=True)
    local_output.unlink(missing_ok=True)
    return result


api = KaggleApi()
api.authenticate()
selected_dates = latest_consecutive_six(api)
print('Latest consecutive six official dates:', selected_dates)
rank_by_team, leaderboard_snapshot = current_top30(api)
print('Leaderboard snapshot:', leaderboard_snapshot)
print('Top30:', sorted((rank, team) for team, rank in rank_by_team.items()))

prep_results = [
    build_day(api, day, rank_by_team, leaderboard_snapshot)
    for day in selected_dates
]

latest_modules = Counter()
modules_by_day: dict[str, Counter[str]] = {}
for result in prep_results:
    counts: Counter[str] = Counter()
    with zipfile.ZipFile(Path(str(result['output']))) as archive:
        rows = list(
            csv.DictReader(
                io.StringIO(
                    archive.read('_index.csv').decode('utf-8-sig')
                )
            )
        )
    counts.update(str(row.get('module_version') or '') for row in rows)
    modules_by_day[str(result['date'])] = counts
    if str(result['date']) == selected_dates[-1]:
        latest_modules = counts
current_module = max(
    latest_modules.items(),
    key=lambda item: (item[1], item[0]),
)[0]
missing = [
    day for day in selected_dates
    if modules_by_day[day][current_module] == 0
]
if missing:
    raise RuntimeError(
        f'Latest module_version={current_module} missing from dates={missing}'
    )

prep_summary = {
    'schema': 'r117_independent_official_replay_prep_v1',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'selected_dates': selected_dates,
    'latest_compatible_module_version': current_module,
    'leaderboard_snapshot': leaderboard_snapshot,
    'days': prep_results,
}
summary_path = REPLAY_ROOT / 'R117_OFFICIAL_REPLAY_PREP_SUMMARY.json'
summary_path.write_text(
    json.dumps(prep_summary, ensure_ascii=False, indent=2),
    encoding='utf-8',
)
print('Six-day input audit READY:', summary_path)


In [ ]:
import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

WORK_ROOT = Path('/content/r117')
if WORK_ROOT.exists():
    shutil.rmtree(WORK_ROOT)
WORK_ROOT.mkdir(parents=True)

with zipfile.ZipFile(PACKAGE_ZIP) as archive:
    root_resolved = WORK_ROOT.resolve()
    for member in archive.infolist():
        destination = (WORK_ROOT / member.filename).resolve()
        if destination != root_resolved and root_resolved not in destination.parents:
            raise RuntimeError(f'Unsafe ZIP member: {member.filename}')
    archive.extractall(WORK_ROOT)

roots = [
    path.parent
    for path in WORK_ROOT.rglob('CODEX_TASK_R117.md')
    if (path.parent / 'run_colab_r117.sh').is_file()
]
assert len(roots) == 1, roots
PROJECT_ROOT = roots[0]

subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        '-r', str(PROJECT_ROOT / 'requirements_gpu.txt'),
    ],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', str(PROJECT_ROOT / 'tests')],
    cwd=PROJECT_ROOT,
    check=True,
)
print('R117 package extracted and static tests passed:', PROJECT_ROOT)


In [ ]:
import os
import torch

assert torch.cuda.is_available(), '请先把 Colab 运行时切换为 GPU，再重新运行'
print('GPU:', torch.cuda.get_device_name(0))

os.environ['R117_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['R117_REPLAY_ROOT'] = str(REPLAY_ROOT)
os.environ['R117_RESULT_ROOT'] = str(RESULT_ROOT)


In [ ]:
import shutil
import subprocess
from datetime import datetime, timezone
from pathlib import Path

LOG_PATH = Path('/content/r117_full.log')
command = ['bash', str(PROJECT_ROOT / 'run_colab_r117.sh')]
print('Starting:', command)
print('Full log:', LOG_PATH)

with LOG_PATH.open('w', encoding='utf-8', buffering=1) as log_handle:
    process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,
        env=os.environ.copy(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log_handle.write(line)
    return_code = process.wait()

log_copy = RESULT_ROOT / (
    'R117_FULL_LOG_' +
    datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') +
    '.log'
)
shutil.copy2(LOG_PATH, log_copy)
print('R117 return code:', return_code)
print('Drive log:', log_copy)
if return_code != 0:
    print(
        'PAUSE: 流水线已按要求停止。不要降低 Gate、不要延长 epoch、'
        '不要把 Test 加入 Train。'
    )


In [ ]:
import json
from pathlib import Path

decision_paths = sorted(
    RESULT_ROOT.rglob('R117_FINAL_DECISION.json'),
    key=lambda path: path.stat().st_mtime,
)
if decision_paths:
    decision_path = decision_paths[-1]
    decision = json.loads(decision_path.read_text(encoding='utf-8'))
    print('FINAL DECISION:', json.dumps(decision, ensure_ascii=False, indent=2))
    print('Decision file:', decision_path)
    if decision.get('status') == 'READY_FOR_FAST_ONLINE_AB':
        archive = Path('/content/PTCG_R117_WGTD_RECOMMENDED.tar.gz')
        print('UPLOAD ALLOWED:', archive)
    else:
        print('UPLOAD FORBIDDEN: Gate 未显示 READY_FOR_FAST_ONLINE_AB')
else:
    pause_paths = sorted(
        RESULT_ROOT.rglob('R117_PAUSE_AND_HANDOFF.md'),
        key=lambda path: path.stat().st_mtime,
    )
    print('No final decision. Latest pause report:', pause_paths[-1] if pause_paths else None)
